[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_57_PaperDistiller_Batch_Eval.ipynb)

# Lesson 57 — paper-distiller: Section Detection, Scanned PDF Fallback, Batch Distiller & Golden Eval Harness

**Phase 6 · OSS Builder** | Prerequisites: Lesson 56 (paper-distiller core pipeline)

---

## What we're building today

In Lesson 56 we built the core `paper-distiller` pipeline: fetch arXiv metadata → extract PDF text → run Claude extraction → generate code example. That gives us a working prototype.

Today we make it **production-worthy** with four upgrades:

| # | Upgrade | Problem Solved |
|---|---------|----------------|
| 1 | **Section Detection** | Claude extracts from a wall of text — if we label sections (Abstract, Method, Results…) first, extraction quality improves significantly |
| 2 | **Scanned PDF Fallback** | ~15% of arXiv PDFs are scanned images — pdfplumber returns empty strings, breaking the pipeline |
| 3 | **Batch Distiller** | Processing papers one-at-a-time is slow; async concurrency lets us distill dozens in parallel |
| 4 | **Golden Eval Harness** | Without measurement, we can't improve; we need a repeatable eval against known-good papers |

## Phase 6 Roadmap

| Lesson | Topic | Status |
|--------|-------|--------|
| 56 | Kickoff + Core Pipeline (FetchLayer, ExtractLayer, CodeLayer) | ✅ Done |
| **57** | **Section Detection + Scanned PDF Fallback + Batch Distiller + Golden Eval** | ⬅️ Today |
| 58 | CLI (Typer), PyPI packaging, `pip install paper-distiller` | 🔜 |
| 59 | GitHub Actions CI, eval gate, automated releases | 🔜 |
| 60 | OSS Polish: README, docs, community health files | 🔜 |

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install anthropic pdfplumber pydantic requests rich nest_asyncio pdf2image Pillow -q

# pdf2image needs poppler system package
!apt-get install -y poppler-utils -q 2>/dev/null || brew install poppler -q 2>/dev/null || echo 'Install poppler manually if not on Colab/Mac'

import anthropic, pdfplumber, requests, base64, io, re, json, asyncio, time
from pathlib import Path
from typing import Optional
from pydantic import BaseModel
from rich.console import Console
from rich.table import Table
import nest_asyncio; nest_asyncio.apply()

# ── API key ────────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import os; ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'sk-ant-...')

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
console = Console()
console.print('[green]✓ Setup complete[/green]')

## Part 1 — Section Detection

### Why section labels matter

When we dump a raw PDF to Claude and ask it to extract findings, we're treating 15 pages as a single undifferentiated block of text. Claude has to infer structure.

If we pre-label sections — **Abstract**, **Introduction**, **Method**, **Results**, **Conclusion** — Claude can focus the right attention on the right parts:

- `one_liner` should come from Abstract
- `method_summary` from the Method/Approach section  
- `key_results` from Results/Experiments
- `limitations` from Discussion or Conclusion

### How section detection works

Academic papers follow a predictable structure. We can detect section boundaries using:

1. **Regex patterns** — lines that are ALL_CAPS or Title Case with known section names
2. **Structural signals** — short lines (< 60 chars) at the start of a paragraph
3. **Numeric prefixes** — `1. Introduction`, `2. Related Work`, etc.

This isn't perfect — and that's fine. Even 70% accurate section labeling helps Claude.

In [ ]:
# ── Part 1: SectionDetector ────────────────────────────────────────────────────

SECTION_PATTERNS = [
    (r'\babstract\b',                       'abstract'),
    (r'\bintroduction\b',                   'introduction'),
    (r'\brelated\s+work\b',                 'related_work'),
    (r'\bbackground\b',                     'background'),
    (r'\b(method|methodology|approach|model|architecture|framework)\b', 'method'),
    (r'\b(experiment|evaluation|empirical\s+study)\b', 'experiments'),
    (r'\b(result|finding)\b',               'results'),
    (r'\b(discussion|analysis)\b',          'discussion'),
    (r'\b(limitation|future\s+work)\b',     'limitations'),
    (r'\bconclusion\b',                     'conclusion'),
    (r'\breference\b',                      'references'),
    (r'\bappendix\b',                       'appendix'),
]

# Pre-compile for speed
COMPILED = [(re.compile(p, re.IGNORECASE), label) for p, label in SECTION_PATTERNS]


def _is_section_header(line: str) -> Optional[str]:
    """Return section label if line looks like a section header, else None."""
    stripped = line.strip()
    if not stripped or len(stripped) > 80:
        return None
    # Must be short and either ALL CAPS, Title Case, or start with a number
    looks_like_header = (
        stripped.isupper()                           # ABSTRACT
        or re.match(r'^\d+\.?\s+[A-Z]', stripped)   # 1. Introduction
        or re.match(r'^[A-Z][a-z]+(?:\s+[A-Za-z]+){0,4}$', stripped)  # Introduction
    )
    if not looks_like_header:
        return None
    for pattern, label in COMPILED:
        if pattern.search(stripped):
            return label
    return None


def detect_sections(full_text: str) -> dict[str, str]:
    """
    Split paper text into labeled sections.
    Returns dict: { section_label -> section_text }
    Text before first detected header goes into 'preamble'.
    """
    sections: dict[str, list[str]] = {}
    current_label = 'preamble'
    sections[current_label] = []

    for line in full_text.split('\n'):
        header = _is_section_header(line)
        if header:
            current_label = header
            if current_label not in sections:
                sections[current_label] = []
        else:
            sections[current_label].append(line)

    # Join and strip each section
    return {k: '\n'.join(v).strip() for k, v in sections.items() if '\n'.join(v).strip()}


def format_sections_for_claude(sections: dict[str, str], max_chars: int = 12000) -> str:
    """
    Format detected sections into a structured string for the extraction prompt.
    Prioritise important sections if we must truncate.
    """
    PRIORITY_ORDER = [
        'abstract', 'method', 'experiments', 'results',
        'conclusion', 'discussion', 'limitations',
        'introduction', 'background', 'related_work',
        'preamble', 'references', 'appendix'
    ]

    ordered = sorted(
        sections.items(),
        key=lambda kv: PRIORITY_ORDER.index(kv[0]) if kv[0] in PRIORITY_ORDER else 99
    )

    parts = []
    total = 0
    for label, text in ordered:
        chunk = f"[{label.upper()}]\n{text[:3000]}"  # per-section cap
        if total + len(chunk) > max_chars:
            break
        parts.append(chunk)
        total += len(chunk)

    return '\n\n'.join(parts)


# ── Demo: section detection on a sample abstract ───────────────────────────────
SAMPLE_TEXT = """
Abstract

We propose a new simple network architecture, the Transformer, based solely on attention mechanisms,
dispensing with recurrence and convolutions entirely.
Experiments on two machine translation tasks show these models to be superior in quality.

1. Introduction

Recurrent neural networks, long short-term memory and gated recurrent neural networks in particular,
have been firmly established as state-of-the-art approaches in sequence modeling.

2. Background

The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU.

3. Model Architecture

The Transformer follows an encoder-decoder structure using stacked self-attention and
point-wise, fully connected layers for both the encoder and decoder.

4. Experiments

We trained on the standard WMT 2014 English-German dataset. Our model achieves 28.4 BLEU.

5. Conclusion

In this work, we presented the Transformer, the first sequence transduction model based entirely
on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures.
"""

sections = detect_sections(SAMPLE_TEXT)
console.print(f"[bold]Detected {len(sections)} sections:[/bold] {list(sections.keys())}")
for label, text in sections.items():
    console.print(f"  [{label}] {len(text)} chars — {text[:80].replace(chr(10), ' ')}...")

print()
formatted = format_sections_for_claude(sections)
console.print(f"[green]Formatted for Claude:[/green] {len(formatted)} chars")

## Part 2 — Scanned PDF Fallback

### The problem with scanned papers

Not all PDFs contain embedded text. Older papers (pre-2000s) or papers from certain publishers are **scanned images** — pixel data stored in a PDF container. `pdfplumber` returns an empty string for each page.

**How to detect a scanned PDF:**
- Extract text with pdfplumber
- If total text is < 200 characters for a multi-page PDF → almost certainly scanned

**Fallback strategy:**
1. Use `pdf2image` to render each PDF page as a PNG (at ~150 DPI for cost efficiency)
2. Encode each page as base64
3. Send page images to Claude vision — Claude can read text from images natively
4. Concatenate Claude's transcriptions as the paper text

**Cost note:** Claude vision charges for image tokens. A 150 DPI page ≈ 1,000 tokens ≈ $0.003 per page with Sonnet. For a 10-page paper: ~$0.03. This is acceptable for a paper distiller where the extracted digest saves hours of reading.

In [ ]:
# ── Part 2: Scanned PDF Fallback ───────────────────────────────────────────────

try:
    from pdf2image import convert_from_bytes
    PDF2IMAGE_AVAILABLE = True
except ImportError:
    PDF2IMAGE_AVAILABLE = False
    console.print('[yellow]pdf2image not available — scanned PDF fallback disabled[/yellow]')


MIN_TEXT_CHARS_PER_PAGE = 100  # below this → likely scanned
VISION_MODEL = 'claude-sonnet-4-5'  # good OCR quality
VISION_DPI = 150  # lower DPI = smaller images = lower cost
MAX_PAGES_FOR_VISION = 12  # cap cost


def is_scanned_pdf(pdf_bytes: bytes) -> bool:
    """Return True if PDF appears to be scanned (low embedded text density)."""
    try:
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            num_pages = len(pdf.pages)
            if num_pages == 0:
                return True
            total_chars = sum(
                len(p.extract_text() or '') for p in pdf.pages[:5]  # check first 5 pages
            )
            chars_per_page = total_chars / min(num_pages, 5)
            return chars_per_page < MIN_TEXT_CHARS_PER_PAGE
    except Exception:
        return True  # treat errors as scanned


def _page_to_base64(page_image) -> str:
    """Convert a PIL Image to base64 PNG string."""
    buf = io.BytesIO()
    page_image.save(buf, format='PNG')
    return base64.standard_b64encode(buf.getvalue()).decode('utf-8')


def extract_text_via_vision(pdf_bytes: bytes, max_pages: int = MAX_PAGES_FOR_VISION) -> tuple[str, float]:
    """
    Use Claude vision to OCR a scanned PDF.
    Returns (extracted_text, estimated_cost_usd).
    """
    if not PDF2IMAGE_AVAILABLE:
        return '(pdf2image unavailable — cannot process scanned PDF)', 0.0

    pages = convert_from_bytes(pdf_bytes, dpi=VISION_DPI)
    pages = pages[:max_pages]

    texts = []
    total_input_tokens = 0

    for i, page in enumerate(pages):
        b64 = _page_to_base64(page)
        resp = client.messages.create(
            model=VISION_MODEL,
            max_tokens=1024,
            messages=[{
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': 'Transcribe ALL text from this academic paper page exactly as written. Preserve paragraph breaks. Output only the transcribed text, nothing else.'
                    }
                ]
            }]
        )
        texts.append(f"--- Page {i+1} ---\n{resp.content[0].text}")
        total_input_tokens += resp.usage.input_tokens

    # Rough cost: Sonnet input $3/MTok
    cost = total_input_tokens * 3.0 / 1_000_000
    return '\n\n'.join(texts), cost


def smart_extract_text(pdf_bytes: bytes) -> tuple[str, str, float]:
    """
    Auto-detect scanned vs native PDF and extract text accordingly.
    Returns (text, method_used, cost_usd).
    """
    if is_scanned_pdf(pdf_bytes):
        console.print('[yellow]⚠ Scanned PDF detected — falling back to Claude vision OCR[/yellow]')
        text, cost = extract_text_via_vision(pdf_bytes)
        return text, 'vision_ocr', cost
    else:
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            text = '\n\n'.join(
                p.extract_text() or '' for p in pdf.pages
            )
        return text, 'pdfplumber', 0.0


# ── Demo: simulate scanned PDF detection ───────────────────────────────────────
# We'll test detection logic with synthetic byte data
console.print('[bold]Scanned PDF Detection Demo[/bold]')
console.print('Real test: fetch 1706.03762 and check extraction method')

def demo_smart_extract(arxiv_id: str):
    url = f'https://arxiv.org/pdf/{arxiv_id}.pdf'
    r = requests.get(url, timeout=30)
    pdf_bytes = r.content
    scanned = is_scanned_pdf(pdf_bytes)
    console.print(f'  arXiv {arxiv_id}: scanned={scanned}')
    if not scanned:
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            total = sum(len(p.extract_text() or '') for p in pdf.pages)
        console.print(f'  → pdfplumber extracted {total:,} chars')

demo_smart_extract('1706.03762')  # Attention Is All You Need — native PDF

# 💡 EXPERIMENT: Try arxiv_id='2005.14165' (GPT-3) — also native
# Some very old papers on arXiv ARE scanned; try searching arxiv.org for pre-1995 papers

## Part 2b — Updated ExtractLayer (with Section Context)

Now we wire `detect_sections` and `smart_extract_text` into the core pipeline from Lesson 56.

In [ ]:
# ── Updated Pipeline Models (from L56) ────────────────────────────────────────
class PaperDigest(BaseModel):
    one_liner: str
    method_summary: str
    key_results: str
    prerequisites: str
    limitations: str
    practitioner_tldr: str
    tags: list[str]
    code_example: str = ''


# ── Tool schema for structured extraction ─────────────────────────────────────
EXTRACT_TOOL = {
    'name': 'extract_digest',
    'description': 'Extract a structured digest from an academic paper',
    'input_schema': {
        'type': 'object',
        'properties': {
            'one_liner': {'type': 'string', 'description': 'One sentence: what this paper does and why it matters'},
            'method_summary': {'type': 'string', 'description': '2-3 sentences on the core technical approach'},
            'key_results': {'type': 'string', 'description': 'Key benchmarks or results with numbers where available'},
            'prerequisites': {'type': 'string', 'description': 'What background does a reader need?'},
            'limitations': {'type': 'string', 'description': 'Honest limitations or failure modes'},
            'practitioner_tldr': {'type': 'string', 'description': 'When should a practitioner use this technique?'},
            'tags': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Up to 5 topic tags'},
        },
        'required': ['one_liner', 'method_summary', 'key_results', 'prerequisites', 'limitations', 'practitioner_tldr', 'tags']
    }
}

EXTRACT_SYSTEM = """You are a senior ML researcher distilling academic papers for practitioners.
Use plain English. Avoid jargon unless necessary. Prefer concrete numbers over vague claims.
The paper text may be labeled with section headers like [ABSTRACT], [METHOD], [RESULTS] — use them."""


def extract_digest(paper_text: str, metadata: dict) -> tuple[PaperDigest, int]:
    """
    Run extraction with section-aware text.
    Returns (PaperDigest, input_tokens_used).
    """
    # Detect sections and format with priority ordering
    sections = detect_sections(paper_text)
    formatted_text = format_sections_for_claude(sections)

    user_msg = f"""Title: {metadata.get('title', 'Unknown')}
Authors: {metadata.get('authors', 'Unknown')}

Paper text (section-labeled):
{formatted_text}"""

    resp = client.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=1024,
        system=EXTRACT_SYSTEM,
        tools=[EXTRACT_TOOL],
        tool_choice={'type': 'tool', 'name': 'extract_digest'},
        messages=[{'role': 'user', 'content': user_msg}]
    )

    tool_input = next(b.input for b in resp.content if b.type == 'tool_use')
    return PaperDigest(**tool_input), resp.usage.input_tokens


console.print('[green]✓ Updated ExtractLayer defined (section-aware)[/green]')
console.print(f'  Sections detected from sample text: {list(sections.keys())}')

## Part 3 — Batch Distiller

### Why sequential processing is too slow

Each paper takes ~5-10 seconds (fetch + extract + codegen). Processing 20 papers sequentially = 100-200 seconds.

With async concurrency, we can run N papers in parallel:
- With semaphore(5): 20 papers ≈ 20-40 seconds (5× speedup)
- Anthropic's rate limit: ~50 requests/min for Sonnet → semaphore(5) is safe

### Batch architecture

```
arxiv_ids list
      │
      ├─→ asyncio.Semaphore(5)  ← rate-limit gate
      │         │
      │    distill_one(id) ← fetch + extract + codegen
      │         │
      └─→ asyncio.gather(*tasks)
                │
          BatchResult (successes + failures + cost total)
                │
          Save to results.jsonl (one JSON per line)
```

We use `.jsonl` (JSON Lines) format — one JSON object per line — because it's appendable, streamable, and easy to parse even if the run is interrupted midway.

In [ ]:
# ── FetchLayer (from L56, needed for BatchDistiller) ──────────────────────────

def parse_arxiv_id(input_str: str) -> str:
    """Extract bare arXiv ID from URL or ID string."""
    input_str = input_str.strip().rstrip('/')
    # Handle abs or pdf URLs
    m = re.search(r'arxiv\.org/(?:abs|pdf)/([\d.]+)', input_str)
    if m:
        return m.group(1)
    # Bare ID like 1706.03762 or 2301.07041v2
    m = re.match(r'^([\d]{4}\.[\d]{4,5}(?:v\d+)?)$', input_str)
    if m:
        return m.group(1)
    raise ValueError(f'Cannot parse arXiv ID from: {input_str}')


def fetch_metadata(arxiv_id: str) -> dict:
    """Fetch paper metadata from arXiv API."""
    clean_id = arxiv_id.split('v')[0]  # strip version
    url = f'http://export.arxiv.org/api/query?id_list={clean_id}'
    r = requests.get(url, timeout=15)
    r.raise_for_status()

    title = re.search(r'<title>([^<]+)</title>', r.text)
    authors = re.findall(r'<name>([^<]+)</name>', r.text)
    summary = re.search(r'<summary>([^<]+)</summary>', r.text, re.DOTALL)

    return {
        'arxiv_id': arxiv_id,
        'title': title.group(1).strip() if title else 'Unknown',
        'authors': ', '.join(authors[:3]) + (' et al.' if len(authors) > 3 else ''),
        'abstract': summary.group(1).strip() if summary else '',
    }


def fetch_pdf_bytes(arxiv_id: str) -> bytes:
    """Download PDF bytes from arXiv."""
    url = f'https://arxiv.org/pdf/{arxiv_id}.pdf'
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.content


console.print('[green]✓ FetchLayer ready[/green]')

In [ ]:
# ── Part 3: BatchDistiller ─────────────────────────────────────────────────────
import dataclasses
from dataclasses import dataclass, field


@dataclass
class DigestResult:
    arxiv_id: str
    title: str = ''
    digest: Optional[PaperDigest] = None
    error: Optional[str] = None
    cost_usd: float = 0.0
    method: str = 'pdfplumber'  # or 'vision_ocr'
    elapsed_s: float = 0.0

    @property
    def success(self) -> bool:
        return self.digest is not None

    def to_dict(self) -> dict:
        d = dataclasses.asdict(self)
        if self.digest:
            d['digest'] = self.digest.model_dump()
        return d


@dataclass
class BatchReport:
    results: list[DigestResult] = field(default_factory=list)

    @property
    def successes(self): return [r for r in self.results if r.success]
    @property
    def failures(self): return [r for r in self.results if not r.success]
    @property
    def total_cost(self): return sum(r.cost_usd for r in self.results)
    @property
    def avg_elapsed(self): return sum(r.elapsed_s for r in self.results) / max(len(self.results), 1)


async def _distill_one(
    arxiv_id: str,
    semaphore: asyncio.Semaphore,
    cost_per_1k_input: float = 0.003  # Sonnet pricing approx
) -> DigestResult:
    """Distill a single paper under semaphore control."""
    async with semaphore:
        t0 = time.time()
        result = DigestResult(arxiv_id=arxiv_id)
        try:
            # Run sync IO in thread pool to not block event loop
            loop = asyncio.get_event_loop()

            meta = await loop.run_in_executor(None, fetch_metadata, arxiv_id)
            result.title = meta['title']

            pdf_bytes = await loop.run_in_executor(None, fetch_pdf_bytes, arxiv_id)

            text, method, ocr_cost = await loop.run_in_executor(
                None, smart_extract_text, pdf_bytes
            )
            result.method = method
            result.cost_usd += ocr_cost

            digest, input_tokens = await loop.run_in_executor(
                None, extract_digest, text, meta
            )
            result.digest = digest
            result.cost_usd += (input_tokens / 1000) * cost_per_1k_input

        except Exception as e:
            result.error = str(e)

        result.elapsed_s = time.time() - t0
        status = '✓' if result.success else '✗'
        console.print(f'  {status} [{arxiv_id}] {result.title[:50]} — {result.elapsed_s:.1f}s ${result.cost_usd:.4f}')
        return result


async def batch_distill(
    arxiv_ids: list[str],
    concurrency: int = 3,
    output_path: Optional[str] = '/content/results.jsonl'
) -> BatchReport:
    """
    Distill multiple papers concurrently.
    Saves results to JSONL (one JSON object per line) — appendable + crash-safe.
    """
    semaphore = asyncio.Semaphore(concurrency)
    console.print(f'[bold]Batch distilling {len(arxiv_ids)} papers (concurrency={concurrency})[/bold]')

    tasks = [_distill_one(aid, semaphore) for aid in arxiv_ids]
    results = await asyncio.gather(*tasks)
    report = BatchReport(results=list(results))

    # Save to JSONL
    if output_path:
        with open(output_path, 'w') as f:
            for r in report.results:
                f.write(json.dumps(r.to_dict()) + '\n')
        console.print(f'[green]Saved to {output_path}[/green]')

    return report


# ── Demo: batch distill 3 landmark papers ─────────────────────────────────────
LANDMARK_PAPERS = [
    '1706.03762',  # Attention Is All You Need
    '1810.04805',  # BERT
    '2106.09685',  # LoRA
]

report = asyncio.run(batch_distill(LANDMARK_PAPERS, concurrency=2))

print()
console.print(f'[bold]Batch Summary[/bold]')
console.print(f'  Successes: {len(report.successes)}/{len(report.results)}')
console.print(f'  Total cost: ${report.total_cost:.4f}')
console.print(f'  Avg time/paper: {report.avg_elapsed:.1f}s')

# Show one digest
if report.successes:
    r = report.successes[0]
    console.print(f'\n[bold]Sample digest — {r.title[:60]}[/bold]')
    console.print(f'  one_liner: {r.digest.one_liner[:120]}')
    console.print(f'  tags: {r.digest.tags}')

# 💡 EXPERIMENT: Change concurrency=5 and add 5 more paper IDs. Observe the speedup.
# 💡 EXPERIMENT: Set output_path='/content/my_batch.jsonl' and read it back with:
#    results = [json.loads(line) for line in open('/content/my_batch.jsonl')]

## Part 4 — Golden Eval Harness

### Why eval matters for OSS

Right now, we assess quality by eye. That doesn't scale, and it's subjective. A **golden eval harness** is:

1. A fixed set of well-known papers where we can verify quality
2. Repeatable metrics we can track across code changes
3. A CI gate — if quality drops below threshold, the PR fails

### What we measure

| Metric | How | Target |
|--------|-----|--------|
| `field_completeness` | All 6 required fields non-empty (0.0–1.0) | ≥ 0.9 |
| `tag_count` | 2–5 tags (0.0 or 1.0) | = 1.0 |
| `one_liner_quality` | LLM judge: 0–3 (specific/concrete/actionable) | ≥ 2.0 |
| `results_has_numbers` | Does `key_results` contain at least one number? | = 1.0 |
| `overall_score` | Weighted average of above | ≥ 0.75 |

### Golden set rationale

We pick papers where the **expected** extraction quality is high because:
- They're well-structured with clear sections
- We know the key results (so we can sanity-check numbers)
- They're native PDFs (not scanned)

If our distiller scores < 0.75 on these, something is wrong.

In [ ]:
# ── Part 4a: Golden Set Definition ────────────────────────────────────────────

GOLDEN_SET = [
    {
        'arxiv_id': '1706.03762',
        'title': 'Attention Is All You Need',
        'expected_tags': ['transformer', 'attention', 'nlp', 'machine translation'],
        'must_mention_in_results': ['BLEU', '28.4'],  # known benchmark numbers
        'must_mention_in_method': ['attention', 'encoder', 'decoder'],
    },
    {
        'arxiv_id': '1810.04805',
        'title': 'BERT',
        'expected_tags': ['bert', 'nlp', 'pre-training', 'fine-tuning'],
        'must_mention_in_results': ['GLUE', 'SQuAD'],
        'must_mention_in_method': ['masked', 'bidirectional'],
    },
    {
        'arxiv_id': '2106.09685',
        'title': 'LoRA',
        'expected_tags': ['lora', 'fine-tuning', 'parameter-efficient'],
        'must_mention_in_results': ['GPT', 'RoBERTa'],
        'must_mention_in_method': ['low-rank', 'adapter'],
    },
]

console.print(f'[bold]Golden set: {len(GOLDEN_SET)} papers[/bold]')
for g in GOLDEN_SET:
    console.print(f"  {g['arxiv_id']} — {g['title']}")

In [ ]:
# ── Part 4b: Evaluation Metrics ───────────────────────────────────────────────

JUDGE_TOOL = {
    'name': 'judge_one_liner',
    'description': 'Score the one_liner field of a paper digest',
    'input_schema': {
        'type': 'object',
        'properties': {
            'score': {
                'type': 'integer',
                'minimum': 0,
                'maximum': 3,
                'description': '0=vague/generic, 1=okay, 2=good/specific, 3=excellent/actionable+concrete'
            },
            'reason': {'type': 'string', 'description': 'One sentence explaining the score'}
        },
        'required': ['score', 'reason']
    }
}


def judge_one_liner(title: str, one_liner: str) -> tuple[int, str]:
    """LLM judge for one_liner quality. Returns (score 0-3, reason)."""
    resp = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=200,
        tools=[JUDGE_TOOL],
        tool_choice={'type': 'tool', 'name': 'judge_one_liner'},
        messages=[{
            'role': 'user',
            'content': f"""Paper: "{title}"
One-liner to evaluate: "{one_liner}"

Score 0-3: Does this one-liner clearly explain what the paper proposes and why it matters to a practitioner?
Score 3 = specific technique named, concrete benefit stated, practitioner knows if they need it."""
        }]
    )
    tool_input = next(b.input for b in resp.content if b.type == 'tool_use')
    return tool_input['score'], tool_input['reason']


def compute_metrics(result: DigestResult, golden: dict) -> dict:
    """Compute all eval metrics for one paper."""
    if not result.success:
        return {'field_completeness': 0, 'tag_count_ok': 0, 'results_has_numbers': 0,
                'one_liner_quality': 0, 'overall_score': 0, 'error': result.error}

    d = result.digest

    # 1. Field completeness: all 6 key fields non-empty
    fields = [d.one_liner, d.method_summary, d.key_results,
              d.prerequisites, d.limitations, d.practitioner_tldr]
    field_completeness = sum(1 for f in fields if len(f.strip()) > 20) / len(fields)

    # 2. Tag count (2-5 is ideal)
    tag_count_ok = 1.0 if 2 <= len(d.tags) <= 5 else 0.0

    # 3. Results has numbers
    results_has_numbers = 1.0 if re.search(r'\d+\.?\d*', d.key_results) else 0.0

    # 4. LLM judge for one_liner
    ql_score, ql_reason = judge_one_liner(result.title, d.one_liner)
    one_liner_quality = ql_score / 3.0  # normalize to 0-1

    # 5. Weighted overall
    overall = (
        0.35 * field_completeness +
        0.15 * tag_count_ok +
        0.20 * results_has_numbers +
        0.30 * one_liner_quality
    )

    return {
        'arxiv_id': result.arxiv_id,
        'title': result.title[:50],
        'field_completeness': round(field_completeness, 2),
        'tag_count_ok': tag_count_ok,
        'results_has_numbers': results_has_numbers,
        'one_liner_quality': round(one_liner_quality, 2),
        'ql_reason': ql_reason,
        'overall_score': round(overall, 2),
    }


console.print('[green]✓ Eval metrics defined[/green]')

In [ ]:
# ── Part 4c: Run Golden Eval Harness ──────────────────────────────────────────

PASS_THRESHOLD = 0.75  # overall_score gate

def run_golden_eval(batch_report: BatchReport, golden_set: list[dict]) -> list[dict]:
    """Run eval on all papers in the batch that match the golden set."""
    golden_by_id = {g['arxiv_id']: g for g in golden_set}
    eval_results = []

    for result in batch_report.results:
        golden = golden_by_id.get(result.arxiv_id)
        if not golden:
            continue
        console.print(f'  Evaluating [{result.arxiv_id}]...')
        metrics = compute_metrics(result, golden)
        eval_results.append(metrics)

    return eval_results


# Run eval on our batch results (from Part 3)
console.print('[bold]Running Golden Eval Harness...[/bold]')
eval_results = run_golden_eval(report, GOLDEN_SET)

# Display results table
print()
table = Table(title='Golden Eval Results', show_lines=True)
table.add_column('Paper', style='cyan', max_width=35)
table.add_column('Fields', justify='center')
table.add_column('Tags', justify='center')
table.add_column('Numbers', justify='center')
table.add_column('1-liner', justify='center')
table.add_column('Overall', justify='center')
table.add_column('Pass?', justify='center')

for m in eval_results:
    passing = m['overall_score'] >= PASS_THRESHOLD
    table.add_row(
        m.get('title', m['arxiv_id'])[:35],
        str(m['field_completeness']),
        '✓' if m['tag_count_ok'] else '✗',
        '✓' if m['results_has_numbers'] else '✗',
        str(m['one_liner_quality']),
        f"[{'green' if passing else 'red'}]{m['overall_score']}[/]",
        '✅' if passing else '❌',
    )

console.print(table)

# Eval gate — this is what CI runs
if eval_results:
    mean_score = sum(m['overall_score'] for m in eval_results) / len(eval_results)
    gate_pass = mean_score >= PASS_THRESHOLD
    color = 'green' if gate_pass else 'red'
    console.print(f'\n[bold {color}]EVAL GATE: mean_score={mean_score:.2f} — {"PASS ✅" if gate_pass else "FAIL ❌"}[/bold {color}]')
    console.print(f'Threshold: {PASS_THRESHOLD}')

# 💡 EXPERIMENT: Lower the threshold to 0.5 and see which papers pass.
# 💡 EXPERIMENT: Add a 4th paper to GOLDEN_SET (e.g. '2005.14165' for GPT-3).

## Part 5 — Writing the Module Files

Now we write these components as proper Python modules inside the `paper-distiller/` project scaffold (from Lesson 56). This is OSS code — it should be clean, documented, and testable.

In [ ]:
# ── Write updated module files ─────────────────────────────────────────────────
BASE = Path('/content/paper_distiller')
BASE.mkdir(parents=True, exist_ok=True)
(BASE / '__init__.py').write_text('')

# ── paper_distiller/section_detector.py ──────────────────────────────────────
(BASE / 'section_detector.py').write_text('''
"""Section detection for academic PDF text."""
import re
from typing import Optional

SECTION_PATTERNS = [
    (r'\\babstract\\b', 'abstract'),
    (r'\\bintroduction\\b', 'introduction'),
    (r'\\brelated\\s+work\\b', 'related_work'),
    (r'\\bbackground\\b', 'background'),
    (r'\\b(method|methodology|approach|model|architecture|framework)\\b', 'method'),
    (r'\\b(experiment|evaluation|empirical\\s+study)\\b', 'experiments'),
    (r'\\b(result|finding)\\b', 'results'),
    (r'\\b(discussion|analysis)\\b', 'discussion'),
    (r'\\b(limitation|future\\s+work)\\b', 'limitations'),
    (r'\\bconclusion\\b', 'conclusion'),
    (r'\\breference\\b', 'references'),
    (r'\\bappendix\\b', 'appendix'),
]
COMPILED = [(re.compile(p, re.IGNORECASE), label) for p, label in SECTION_PATTERNS]

PRIORITY_ORDER = [
    'abstract', 'method', 'experiments', 'results',
    'conclusion', 'discussion', 'limitations',
    'introduction', 'background', 'related_work',
    'preamble', 'references', 'appendix'
]


def detect_sections(full_text: str) -> dict[str, str]:
    sections: dict[str, list[str]] = {}
    current_label = \'preamble\'
    sections[current_label] = []
    for line in full_text.split(\'\\n\'):
        stripped = line.strip()
        header = None
        if stripped and len(stripped) <= 80 and (
            stripped.isupper() or
            re.match(r\'^\'\\d+\\.?\\s+[A-Z]\', stripped) or
            re.match(r\'^[A-Z][a-z]+(?:\\s+[A-Za-z]+){0,4}$\', stripped)
        ):
            for pattern, label in COMPILED:
                if pattern.search(stripped):
                    header = label; break
        if header:
            current_label = header
            if current_label not in sections:
                sections[current_label] = []
        else:
            sections[current_label].append(line)
    return {k: \'\\n\'.join(v).strip() for k, v in sections.items() if \'\\n\'.join(v).strip()}


def format_sections_for_claude(sections: dict[str, str], max_chars: int = 12000) -> str:
    ordered = sorted(
        sections.items(),
        key=lambda kv: PRIORITY_ORDER.index(kv[0]) if kv[0] in PRIORITY_ORDER else 99
    )
    parts, total = [], 0
    for label, text in ordered:
        chunk = f\'[{label.upper()}]\\n{text[:3000]}\'
        if total + len(chunk) > max_chars:
            break
        parts.append(chunk); total += len(chunk)
    return \'\\n\\n\'.join(parts)
''')

# ── paper_distiller/fallback.py ───────────────────────────────────────────────
(BASE / 'fallback.py').write_text('''
"""Scanned PDF detection and Claude vision fallback."""
import io, base64
import pdfplumber
import anthropic

MIN_CHARS_PER_PAGE = 100
MAX_PAGES = 12
VISION_DPI = 150


def is_scanned(pdf_bytes: bytes) -> bool:
    try:
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            if not pdf.pages: return True
            total = sum(len(p.extract_text() or \'\') for p in pdf.pages[:5])
            return total / min(len(pdf.pages), 5) < MIN_CHARS_PER_PAGE
    except Exception:
        return True


def extract_via_vision(pdf_bytes: bytes, client: anthropic.Anthropic) -> tuple[str, float]:
    try:
        from pdf2image import convert_from_bytes
    except ImportError:
        return \'(pdf2image unavailable)\', 0.0
    pages = convert_from_bytes(pdf_bytes, dpi=VISION_DPI)[:MAX_PAGES]
    texts, tokens = [], 0
    for i, page in enumerate(pages):
        buf = io.BytesIO()
        page.save(buf, format=\'PNG\')
        b64 = base64.standard_b64encode(buf.getvalue()).decode()
        resp = client.messages.create(
            model=\'claude-sonnet-4-5\', max_tokens=1024,
            messages=[{\'role\': \'user\', \'content\': [
                {\'type\': \'image\', \'source\': {\'type\': \'base64\', \'media_type\': \'image/png\', \'data\': b64}},
                {\'type\': \'text\', \'text\': \'Transcribe ALL text from this academic paper page exactly.\'}
            ]}]
        )
        texts.append(f\'--- Page {i+1} ---\\n{resp.content[0].text}\')
        tokens += resp.usage.input_tokens
    return \'\\n\\n\'.join(texts), tokens * 3.0 / 1_000_000
''')

# ── paper_distiller/batch.py ──────────────────────────────────────────────────
(BASE / 'batch.py').write_text('''
"""Batch distiller: async concurrent paper processing."""
import asyncio, json, time
from dataclasses import dataclass, field
from typing import Optional
from pathlib import Path


@dataclass
class BatchReport:
    results: list = field(default_factory=list)

    @property
    def successes(self): return [r for r in self.results if r.get(\'success\')]
    @property
    def failures(self): return [r for r in self.results if not r.get(\'success\')]
    @property
    def total_cost(self): return sum(r.get(\'cost_usd\', 0) for r in self.results)
''')

# ── evals/golden_harness.py ───────────────────────────────────────────────────
EVALS_DIR = BASE / 'evals'
EVALS_DIR.mkdir(exist_ok=True)
(EVALS_DIR / '__init__.py').write_text('')
(EVALS_DIR / 'golden_harness.py').write_text(f'''
"""Golden eval harness for paper-distiller."""
GOLDEN_SET = {json.dumps(GOLDEN_SET, indent=2)}
PASS_THRESHOLD = 0.75

def eval_gate(eval_results: list[dict]) -> bool:
    """Return True if mean overall_score >= PASS_THRESHOLD."""
    if not eval_results: return False
    mean = sum(m[\'overall_score\'] for m in eval_results) / len(eval_results)
    return mean >= PASS_THRESHOLD
''')

# Show what we created
import os
for root, dirs, files in os.walk(BASE):
    level = root.replace(str(BASE), '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    print(f'{indent}📁 {folder}/')
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        print(f'{indent}  📄 {f} ({size} bytes)')

## 10 Pitfalls to Avoid

| # | Pitfall | Why it hurts | Fix |
|---|---------|-------------|-----|
| 1 | **Trusting section detector blindly** | Papers use non-standard headers; 30% may not match | Fall back to full-text if < 2 sections detected |
| 2 | **Running vision OCR on native PDFs** | 10× the cost, no benefit | Always check `is_scanned()` first |
| 3 | **No semaphore on batch** | Rate-limit 429 errors flood in | Use `asyncio.Semaphore(3–5)` |
| 4 | **asyncio.gather fails fast on first error** | One bad paper kills the batch | Pass `return_exceptions=True` and handle per-result |
| 5 | **JSONL not crash-safe** | File written all-at-once → lost on crash | Write each result immediately after completion |
| 6 | **Eval on training papers** | LLM memorised them — artificially high scores | Use golden set with less-famous papers for accuracy |
| 7 | **LLM judge on every CI run** | Expensive + slow CI | Cache judge results; re-run only when extract logic changes |
| 8 | **Threshold too high too early** | Eval gate blocks all PRs before tuning | Start at 0.60, raise by 0.05 per iteration |
| 9 | **PDF bytes in memory for 50 papers** | OOM on Colab (12GB RAM) | Fetch, process, discard bytes before next paper |
| 10 | **Scanned fallback without page cap** | 50-page paper = $0.15 in vision tokens | Always cap at `MAX_PAGES = 12` |

---

**Pitfall #4 — live demo:**

In [ ]:
# ── Pitfall #4: gather fails fast — fix with return_exceptions=True ────────────

async def _fake_task(n: int):
    await asyncio.sleep(0.1 * n)
    if n == 2:
        raise ValueError('Paper #2 failed!')
    return f'paper_{n}_done'

# BAD: one failure = entire batch lost
try:
    bad_results = await asyncio.gather(*[_fake_task(i) for i in range(5)])
except Exception as e:
    console.print(f'[red]BAD pattern: batch killed by exception: {e}[/red]')
    console.print('  → Papers 3, 4 never completed!')

print()

# GOOD: return_exceptions=True — failures are values, not interrupts
good_results = await asyncio.gather(*[_fake_task(i) for i in range(5)], return_exceptions=True)
for i, r in enumerate(good_results):
    if isinstance(r, Exception):
        console.print(f'[yellow]  Paper {i}: FAILED — {r}[/yellow]')
    else:
        console.print(f'[green]  Paper {i}: {r}[/green]')

console.print('\n[green]✓ GOOD pattern: 4/5 papers succeed despite one failure[/green]')

## Summary

Today you built 4 production upgrades to `paper-distiller`:

| Component | What it does | Key function |
|-----------|-------------|-------------|
| `SectionDetector` | Labels Abstract/Method/Results/… from raw text | `detect_sections()` |
| `ScannedFallback` | Detects scanned PDFs, OCRs via Claude vision | `smart_extract_text()` |
| `BatchDistiller` | Processes N papers in parallel with semaphore | `batch_distill()` |
| `GoldenEvalHarness` | Measures extraction quality with LLM judge + CI gate | `run_golden_eval()` |

**The key insight:** Quality in ML engineering is about measurement. Without the eval harness, you're guessing. With it, every refactor either passes or fails — empirically.

---

## Homework (5 exercises)

1. **Fuzzy section matching**: Some papers write `"III. METHODOLOGY"` or `"§4 Experiments"`. Add 3 more regex patterns to catch these.

2. **JSONL streaming**: Modify `batch_distill` to write each result to the JSONL file *immediately* when complete (not all at once at the end). This makes it crash-safe — if Colab disconnects at paper #8, papers 1–7 are already saved.

3. **Cost budget**: Add a `max_cost_usd: float = 1.0` parameter to `batch_distill`. After each paper, check total cost so far. If it exceeds the budget, cancel remaining tasks with `task.cancel()`.

4. **Expand the golden set**: Add 2 more papers to `GOLDEN_SET`. Good candidates: `2005.14165` (GPT-3), `2302.13971` (LLaMA). Run the eval and see how the scores differ from the original 3.

5. **CI script**: Write a Python script `evals/run_ci_eval.py` that: (1) reads `results.jsonl`, (2) runs `run_golden_eval()`, (3) calls `eval_gate()`, (4) exits with `sys.exit(1)` if the gate fails. This is what your GitHub Actions workflow will call.

---

## Coming up in Lesson 58

**CLI + PyPI Packaging**: We'll build a `paper-distiller` CLI using Typer so users can run:
```bash
pip install paper-distiller
paper-distiller distill 1706.03762
paper-distiller batch ids.txt --output results.jsonl
```
Then we'll publish to TestPyPI and make it a real installable package.